# 1. Objetivos do estudo  

O objetivo deste estudo é analisar o banco de dados de um serviço de livros para compreender o comportamento de publicação, avaliação e interação dos usuários com os livros.  

O banco de dados é composto pelas tabelas **books, authors, publishers, ratings e reviews**.

A análise será realizada considerando a relação entre as diferentes tabelas que compõem o banco de dados. A tabela books contém as principais informações sobre as obras, como título, número de páginas e data de publicação, e funciona como elemento central para a integração das informações. Cada livro está relacionado a um autor por meio do campo author_id, estabelecendo uma relação entre as tabelas books e authors. Essa relação permite identificar quais autores estão associados a cada obra e analisar o desempenho de seus livros.

Da mesma forma, a tabela books está relacionada à tabela publishers por meio do campo publisher_id. Essa relação permite identificar a editora responsável por cada livro e, consequentemente, analisar a quantidade de obras publicadas por cada editora.

A tabela ratings registra as classificações atribuídas pelos usuários aos livros. Ela se relaciona com a tabela books por meio do campo book_id, permitindo associar cada avaliação ao livro correspondente. A partir dessa relação, é possível calcular a quantidade de classificações recebidas por cada livro, sua classificação média e analisar a interação dos usuários com as diferentes obras.

A tabela ratings também contém a identificação dos usuários por meio do campo username. Dessa forma, é possível agrupar as avaliações por usuário e analisar o comportamento dos leitores, como a quantidade de livros diferentes avaliados por cada usuário. Essa informação permite identificar o nível de engajamento dos usuários mais ativos da plataforma.

A tabela books também se relaciona com a tabela reviews por meio do campo book_id. Enquanto a tabela ratings registra as classificações numéricas atribuídas pelos usuários, a tabela reviews armazena as avaliações ou comentários escritos sobre os livros. Essa relação permite complementar a análise quantitativa das classificações com informações provenientes das avaliações dos leitores, possibilitando uma compreensão mais ampla da interação dos usuários com as obras.

Essa estrutura permite combinar informações sobre autores, livros, editoras e usuários, possibilitando uma análise integrada do catálogo e das interações realizadas na plataforma. Assim, as consultas SQL utilizam os relacionamentos entre as tabelas para transformar os dados armazenados em informações relevantes sobre as publicações, a percepção dos leitores e o nível de engajamento dos usuários.

A análise busca responder às seguintes questões:  

**1. Identificar o número de livros publicados após 1º de janeiro de 2000**, permitindo compreender o volume de publicações mais recentes disponíveis na base.  
**2. Calcular o número de avaliações e a classificação média de cada livro**, para identificar o nível de interação dos usuários e o desempenho dos livros segundo suas classificações.  
**3. Identificar a editora que publicou o maior número de livros com mais de 50 páginas**, excluindo as publicações menores.  
**4. Identificar o autor cujos livros apresentam a maior classificação média**, considerando apenas autores com livros que tenham recebido pelo menos 50 classificações, garantindo uma base mínima de avaliações para a comparação.  
**5. Calcular o número médio de livros avaliados pelos usuários que avaliaram mais de 50 livros**, permitindo analisar o nível de engajamento dos usuários mais ativos da plataforma.  

Os resultados dessas análises serão utilizados para compreender melhor o catálogo de livros, o desempenho das publicações e o comportamento dos usuários, fornecendo informações que podem contribuir para a elaboração de uma proposta para um novo produto no mercado de livros.

# 2. Exploração inicial do banco de dados

Antes de realizar as análises, será feita uma exploração inicial das tabelas disponíveis no banco de dados, primeiramente importando as bibliotecas e conectando-se ao banco de dados.

In [1]:
# import libraries
import pandas as pd
from sqlalchemy import create_engine
db_config = {
 'user': 'practicum_student', # username
 'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7', # password
 'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
 'port': 5432, # connection port
 'db': 'data-analyst-final-project-db' # the name of the database
 }
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],

db_config['pwd'],

db_config['host'],
db_config['port'],

db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [2]:
# Imprimindo as primeiras linhas da tabela books
query = """
SELECT *
FROM books
LIMIT 5;
"""

pd.read_sql(query, engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


A tabela books contém as informações principais dos livros, incluindo identificador, autor, título, número de páginas, data de publicação e editora.

In [3]:
# Imprimindo as primeiras linhas da tabela authors
query = """
SELECT *
FROM authors
LIMIT 5;
"""

pd.read_sql(query, engine)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


A tabela authors contém o identificador de cada autor e seu respectivo nome. O campo author_id é utilizado para relacionar os autores aos livros presentes na tabela books.

In [4]:
# Imprimindo as primeiras linhas da tabela publishers
query = """
SELECT *
FROM publishers
LIMIT 5;
"""

pd.read_sql(query, engine)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


A tabela publishers contém o identificador e o nome das editoras. O campo publisher_id permite relacionar cada editora aos livros publicados por ela na tabela books.

In [5]:
# Imprimindo as primeiras linhas da tabela ratings
query = """
SELECT *
FROM ratings
LIMIT 5;
"""

pd.read_sql(query, engine)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


A tabela ratings registra as classificações atribuídas pelos usuários aos livros. Cada registro possui um identificador da classificação, o livro avaliado, o usuário responsável pela classificação e a nota atribuída. O campo book_id permite relacionar essa tabela com a tabela books.

In [6]:
# Imprimindo as primeiras linhas da tabela reviews
query = """
SELECT *
FROM reviews
LIMIT 5;
"""

pd.read_sql(query, engine)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


A tabela reviews contém as avaliações textuais feitas pelos usuários. Cada registro identifica a avaliação, o livro avaliado, o usuário responsável e o texto da avaliação. A coluna book_id permite relacionar essa tabela com a tabela books e a coluna username permite relacionar essa tabela com a tabela ratings.

In [7]:
# Verificando o tamanho de cada tabela
query = """
SELECT 'books' AS table_name, COUNT(*) AS rows
FROM books

UNION ALL

SELECT 'authors' AS table_name, COUNT(*) AS rows
FROM authors

UNION ALL

SELECT 'publishers' AS table_name, COUNT(*) AS rows
FROM publishers

UNION ALL

SELECT 'ratings' AS table_name, COUNT(*) AS rows
FROM ratings

UNION ALL

SELECT 'reviews' AS table_name, COUNT(*) AS rows
FROM reviews;
"""

pd.read_sql(query, engine)

,table_name,rows
0,books,1000
1,authors,636
2,publishers,340
3,ratings,6456
4,reviews,2793


# 3. Realização das consultas SQL para cada uma das tarefas

In [8]:
# Encontrando o número de livros lançados depois de 1 de janeiro de 2000
query = """
SELECT COUNT(*) AS number_of_books
FROM books
WHERE publication_date > '2000-01-01';
"""

result = pd.read_sql(query, engine)

result

,number_of_books
0,819


Foram publicados 819 livros depois de 1 de janeiro de 2000.

In [9]:
# Encontrando o número de avaliações e a classificação média para cada livro
query = """
SELECT
    book_id,
    COUNT(rating_id) AS number_of_ratings,
    AVG(rating) AS average_rating
FROM ratings
GROUP BY book_id
ORDER BY book_id;
"""

result = pd.read_sql(query, engine)

result

,book_id,number_of_ratings,average_rating
0,1,3,3.666667
1,2,2,2.500000
2,3,3,4.666667
3,4,2,4.500000
4,5,6,4.000000
...,...,...,...
995,996,3,3.666667
996,997,5,3.400000
997,998,5,3.200000
998,999,2,4.500000


A consulta permitiu identificar, para cada livro, o número de classificações recebidas e sua classificação média. Dessa forma, é possível observar tanto o nível de interação dos usuários com cada livro quanto a percepção geral dos leitores em relação à qualidade das obras. Vale salientar que apenas uma classificação média maior não significa que um livro é melhor, já que a quantidade de classificações também deve ser considerada. Por exemplo, um livro com média 4.67 baseada em poucas avaliações tem uma evidência menor do que um livro com média 4.50 baseado em muitas classificações.

In [10]:
# Identificando a editora que lançou o maior número de livros com mais de 50 páginas
query = """
SELECT
    p.publisher,
    COUNT(b.book_id) AS number_of_books
FROM books AS b
JOIN publishers AS p
    ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY number_of_books DESC
LIMIT 1;
"""

result = pd.read_sql(query, engine)

result

,publisher,number_of_books
0,Penguin Books,42


A editora que publicou o maior número de livros com mais de 50 páginas foi a  Penguin Books, com 42 livros. O filtro de mais de 50 páginas ajuda a excluir publicações muito curtas, como brochuras e materiais semelhantes, tornando a análise do catálogo mais relevante para livros completos.

In [11]:
# Identificando o autor com a média mais alta de classificação de livros, considerando apenas livros que tenham pelo menos 50 classificações
query = """
SELECT
    a.author,
    AVG(r.rating) AS average_rating
FROM ratings AS r
JOIN books AS b
    ON r.book_id = b.book_id
JOIN authors AS a
    ON b.author_id = a.author_id
WHERE r.book_id IN (
    SELECT book_id
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)
GROUP BY a.author
ORDER BY average_rating DESC
LIMIT 1;
"""

result = pd.read_sql(query, engine)

result

,author,average_rating
0,J.K. Rowling/Mary GrandPré,4.287097


Considerando somente livros que receberam pelo menos 50 classificações, o autor com a maior classificação média foi J.K. Rowling/Mary GrandPré, com média de 4.29. O critério de pelo menos 50 classificações reduz o impacto de livros com poucas avaliações.

In [12]:
# Encontrando o número médio de avaliações entre usuários que classificaram mais do que 50 livros
query = """
SELECT
    AVG(number_of_ratings) AS average_number_of_ratings
FROM (
    SELECT
        username,
        COUNT(DISTINCT book_id) AS number_of_ratings
    FROM ratings
    GROUP BY username
    HAVING COUNT(DISTINCT book_id) > 50
) AS user_ratings;
"""

result = pd.read_sql(query, engine)

result

,average_number_of_ratings
0,54.333333


Entre os usuários que classificaram mais de 50 livros, a média foi de 54.33 livros classificados por usuário. Esse resultado indica o nível de engajamento dos usuários mais ativos da plataforma, mostrando que esse grupo possui uma participação significativamente maior no sistema de classificações.

In [13]:
# Encontrando o número médio de resenhas de texto entre os usuários que classificaram mais do que 50 livros
query = """
SELECT
    AVG(review_count) AS avg_reviews
FROM (
    SELECT
        r.username,
        COUNT(rv.review_id) AS review_count
    FROM (
        SELECT
            username
        FROM ratings
        GROUP BY username
        HAVING COUNT(book_id) > 50
    ) AS r
    LEFT JOIN reviews AS rv
        ON r.username = rv.username
    GROUP BY r.username
) AS user_reviews;
"""

result = pd.read_sql(query, engine)

result

,avg_reviews
0,24.333333


O número médio de resenhas de texto entre os usuários que classificaram mais de 50 livros foi de 24.33 resenhas por usuário. Isso indica que embora esses usuários sejam bastante ativos na classificação de livros (mais de 50 classificações), eles não necessariamente escrevem uma resenha para cada livro que avaliam. Portanto, o número de classificações e o número de resenhas representam comportamentos distintos dos usuários na plataforma.

# 4. Conclusões gerais

A análise realizada permitiu obter uma visão abrangente sobre o catálogo de livros, a interação dos usuários com as obras e o desempenho das publicações na plataforma. Foram identificados 819 livros publicados após 1 de janeiro de 2000, demonstrando que o banco de dados possui uma quantidade relevante de obras mais recentes, o que pode ser interessante para avaliar tendências e preferências dos leitores ao longo do tempo.

A análise das classificações também mostrou que a média de avaliações deve ser interpretada em conjunto com o número de classificações recebidas. Uma média elevada baseada em poucas avaliações apresenta menor evidência sobre a aceitação geral de uma obra do que uma média semelhante obtida a partir de um grande número de usuários. Dessa forma, tanto a popularidade, representada pelo volume de classificações, quanto a avaliação média, relacionada à percepção dos leitores, são métricas importantes para avaliar o desempenho dos livros.

Em relação às editoras, a Penguin Books se destacou como a editora com o maior número de livros com mais de 50 páginas, totalizando 42 obras. Esse resultado indica uma presença relevante da editora no catálogo analisado, considerando um critério que prioriza publicações mais substanciais e reduz a influência de materiais muito curtos.

Entre os autores cujos livros receberam pelo menos 50 classificações, J.K. Rowling/Mary GrandPré apresentou a maior classificação média, de 4.29. O requisito mínimo de 50 avaliações torna essa comparação mais confiável, pois reduz a possibilidade de que médias muito altas sejam resultado de um número reduzido de opiniões. Esse resultado pode indicar uma forte aceitação das obras do autor entre os usuários da plataforma.

O comportamento dos usuários revelou um grupo de leitores bastante engajado. Os usuários que classificaram mais de 50 livros avaliaram, em média, 54.33 livros por pessoa. Esse nível de participação demonstra que existe uma parcela de usuários com histórico significativo de interação com a plataforma. Esses usuários podem ser especialmente importantes para um novo produto, pois possuem maior quantidade de dados sobre suas preferências, o que pode favorecer a criação de sistemas de recomendação mais personalizados.

Por fim, o número médio de resenhas de texto entre os usuários que classificaram mais de 50 livros foi de 24.33 resenhas por usuário. Isso indica que embora esses usuários sejam bastante ativos na classificação de livros (mais de 50 classificações), eles não necessariamente escrevem uma resenha para cada livro que avaliam. Portanto, o número de classificações e o número de resenhas representam comportamentos distintos dos usuários na plataforma.

De maneira geral, os resultados indicam que, para a elaboração de uma proposta de novo produto no mercado de livros, é importante considerar conjuntamente a qualidade percebida, a popularidade das obras, a presença das editoras, o desempenho dos autores e o nível de engajamento dos usuários. A combinação dessas informações pode contribuir para identificar livros e autores com maior potencial, compreender melhor os interesses dos leitores e desenvolver funcionalidades baseadas em recomendações personalizadas. Além disso, o comportamento dos usuários mais ativos sugere uma oportunidade de utilizar o histórico de classificações para oferecer experiências mais relevantes e aumentar o engajamento na plataforma.

Assim, os resultados não apenas descrevem o catálogo existente, mas também fornecem indicadores que podem apoiar decisões sobre seleção de conteúdo, recomendações de livros e estratégias para aumentar a interação dos usuários com um novo produto.